# Data Visualization — Matplotlib & Seaborn

If you can't see it, you can't understand it. Visualization is the bridge between raw numbers and insight.  
We'll use **Matplotlib** (the workhorse) and **Seaborn** (the stylist).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
rng = np.random.default_rng(42)

n = 200
df = pd.DataFrame({
    'hours_studied': rng.uniform(1, 10, n).round(1),
    'sleep_hours': rng.normal(7, 1.5, n).clip(3, 12).round(1),
    'exam_score': np.nan,
    'department': rng.choice(['CS', 'Math', 'Physics', 'Biology'], n),
    'year': rng.choice([1, 2, 3, 4], n)
})
noise = rng.normal(0, 8, n)
df['exam_score'] = (8 * df['hours_studied'] + 3 * df['sleep_hours'] + 15 + noise).clip(0, 100).round(1)
df['passed'] = df['exam_score'] >= 60

print(f'Dataset: {n} students')
df.head()

---
## Matplotlib Basics — The Object-Oriented API

Always use `fig, ax = plt.subplots()`. It gives you full control.  
Avoid the `plt.plot()` shortcut — it works for quick stuff but breaks for complex figures.

In [ ]:
months = np.arange(1, 13)
revenue = [42, 45, 50, 48, 55, 60, 58, 62, 70, 68, 75, 80]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(months, revenue, color='steelblue', linewidth=2, marker='o', markersize=6)
ax.set_xlabel('Month')
ax.set_ylabel('Revenue ($K)')
ax.set_title('Monthly Revenue — 2024')
ax.set_xticks(months)
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                     'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Line Plots — Trends Over Time

Use when you have a **continuous x-axis** (time, sequence) and want to show trends.

In [ ]:
days = np.arange(1, 31)
product_a = 100 + np.cumsum(rng.normal(2, 5, 30))
product_b = 120 + np.cumsum(rng.normal(1.5, 4, 30))
product_c = 80 + np.cumsum(rng.normal(3, 6, 30))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(days, product_a, label='Product A', linewidth=2, color='#e74c3c')
ax.plot(days, product_b, label='Product B', linewidth=2, color='#3498db', linestyle='--')
ax.plot(days, product_c, label='Product C', linewidth=2, color='#2ecc71', linestyle='-.')
ax.fill_between(days, product_a, alpha=0.1, color='#e74c3c')
ax.set_xlabel('Day of Month')
ax.set_ylabel('Cumulative Sales')
ax.set_title('Daily Cumulative Sales by Product')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Bar Charts — Comparing Categories

Use for **categorical** data. Vertical for few categories, horizontal for many or long labels.

In [ ]:
dept_stats = df.groupby('department').agg(
    avg_score=('exam_score', 'mean'),
    count=('exam_score', 'count')
).round(1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']
axes[0].bar(dept_stats.index, dept_stats['avg_score'], color=colors, edgecolor='white')
axes[0].set_ylabel('Average Exam Score')
axes[0].set_title('Average Score by Department')
for i, v in enumerate(dept_stats['avg_score']):
    axes[0].text(i, v + 0.5, f'{v:.1f}', ha='center', fontweight='bold')

axes[1].barh(dept_stats.index, dept_stats['count'], color=colors, edgecolor='white')
axes[1].set_xlabel('Number of Students')
axes[1].set_title('Students per Department')

plt.tight_layout()
plt.show()

In [ ]:
pivot = df.groupby(['department', 'year']).size().unstack(fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pivot.plot(kind='bar', ax=axes[0], width=0.8)
axes[0].set_title('Grouped Bar — Students per Dept & Year')
axes[0].set_ylabel('Count')
axes[0].legend(title='Year')
axes[0].tick_params(axis='x', rotation=0)

pivot.plot(kind='bar', stacked=True, ax=axes[1], width=0.8)
axes[1].set_title('Stacked Bar — Same Data, Different Story')
axes[1].set_ylabel('Count')
axes[1].legend(title='Year')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

---
## Histograms — Distribution of Values

Use when you want to see **how data is spread**. Always adjust `bins`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df['exam_score'], bins=10, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('10 bins')
axes[0].set_xlabel('Exam Score')

axes[1].hist(df['exam_score'], bins=25, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].set_title('25 bins — better resolution')
axes[1].set_xlabel('Exam Score')

for dept in ['CS', 'Math']:
    axes[2].hist(df[df['department'] == dept]['exam_score'],
                 bins=20, alpha=0.5, label=dept, edgecolor='white')
axes[2].set_title('Overlaid — CS vs Math')
axes[2].set_xlabel('Exam Score')
axes[2].legend()

plt.tight_layout()
plt.show()

---
## Scatter Plots — Relationships Between Variables

Use to explore **correlations**. Color and size add extra dimensions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df['hours_studied'], df['exam_score'], alpha=0.5, s=30, color='steelblue')
axes[0].set_xlabel('Hours Studied')
axes[0].set_ylabel('Exam Score')
axes[0].set_title('Study Hours vs Exam Score')

z = np.polyfit(df['hours_studied'], df['exam_score'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['hours_studied'].min(), df['hours_studied'].max(), 100)
axes[0].plot(x_line, p(x_line), 'r--', linewidth=2, label=f'Trend (slope={z[0]:.1f})')
axes[0].legend()

dept_colors = {'CS': '#e74c3c', 'Math': '#3498db', 'Physics': '#2ecc71', 'Biology': '#f39c12'}
for dept, color in dept_colors.items():
    mask = df['department'] == dept
    axes[1].scatter(df[mask]['hours_studied'], df[mask]['exam_score'],
                    alpha=0.6, s=df[mask]['sleep_hours'] * 10,
                    color=color, label=dept, edgecolors='white', linewidth=0.5)
axes[1].set_xlabel('Hours Studied')
axes[1].set_ylabel('Exam Score')
axes[1].set_title('By Department (size = sleep hours)')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## Box Plots — Distribution Summary

Shows median, quartiles, and outliers at a glance. Great for **comparing groups**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

df.boxplot(column='exam_score', by='department', ax=axes[0])
axes[0].set_title('Exam Score by Department')
axes[0].set_xlabel('Department')
axes[0].set_ylabel('Score')
fig.suptitle('')

df.boxplot(column='exam_score', by='year', ax=axes[1])
axes[1].set_title('Exam Score by Year')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Score')

plt.tight_layout()
plt.show()

---
## Heatmaps — Correlation & Matrices

Visualize relationships between **all pairs** of numeric variables at once.

In [ ]:
numeric_cols = df.select_dtypes(include=np.number)
corr = numeric_cols.corr().round(2)

fig, ax = plt.subplots(figsize=(8, 6))
im = sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
                 square=True, linewidths=1, ax=ax, fmt='.2f')
ax.set_title('Correlation Matrix')
plt.tight_layout()
plt.show()

---
## Subplots — Multiple Plots in One Figure

Use `plt.subplots(rows, cols)` to create a grid. Index axes like a 2D array.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].hist(df['exam_score'], bins=20, color='steelblue', edgecolor='white')
axes[0, 0].set_title('Distribution of Scores')
axes[0, 0].set_xlabel('Score')

axes[0, 1].scatter(df['hours_studied'], df['exam_score'], alpha=0.4, s=20, c='coral')
axes[0, 1].set_title('Study vs Score')
axes[0, 1].set_xlabel('Hours')
axes[0, 1].set_ylabel('Score')

dept_means = df.groupby('department')['exam_score'].mean()
axes[1, 0].bar(dept_means.index, dept_means.values, color=['#e74c3c', '#3498db', '#2ecc71', '#f39c12'])
axes[1, 0].set_title('Avg Score by Dept')
axes[1, 0].set_ylabel('Score')

sns.boxplot(data=df, x='year', y='exam_score', ax=axes[1, 1], palette='Set2')
axes[1, 1].set_title('Score by Year')

fig.suptitle('Student Performance Dashboard', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## Seaborn — Statistical Visualization Made Easy

Seaborn wraps Matplotlib with **sane defaults** and **statistical awareness**.  
One line of Seaborn = 10 lines of Matplotlib.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.countplot(data=df, x='department', hue='passed', ax=axes[0, 0], palette='Set2')
axes[0, 0].set_title('Pass/Fail by Department')

sns.violinplot(data=df, x='department', y='exam_score', ax=axes[0, 1],
               palette='muted', inner='quartile')
axes[0, 1].set_title('Score Distribution (Violin)')

sns.regplot(data=df, x='hours_studied', y='exam_score', ax=axes[1, 0],
            scatter_kws={'alpha': 0.3, 's': 20}, color='steelblue')
axes[1, 0].set_title('Study Hours vs Score (with regression)')

sns.kdeplot(data=df, x='exam_score', hue='department', ax=axes[1, 1], fill=True, alpha=0.3)
axes[1, 1].set_title('Score Density by Department')

plt.tight_layout()
plt.show()

In [ ]:
plot_df = df[['hours_studied', 'sleep_hours', 'exam_score']].copy()
plot_df['department'] = df['department']

g = sns.pairplot(plot_df, hue='department', height=2.5, palette='Set1',
                 plot_kws={'alpha': 0.5, 's': 20})
g.figure.suptitle('Pairplot — All Variable Relationships', y=1.02)
plt.show()

---
## Plot Types Decision Guide

| I want to show... | Use this plot | Example |
|---|---|---|
| **Trend over time** | Line plot | Revenue by month |
| **Compare categories** | Bar chart | Sales by region |
| **Distribution** | Histogram / KDE | Test score spread |
| **Relationship (2 vars)** | Scatter plot | Height vs Weight |
| **Distribution + outliers** | Box plot / Violin | Salary by department |
| **All correlations** | Heatmap | Feature correlations |
| **Composition** | Stacked bar / Pie | Market share |
| **All pairwise** | Pairplot | Multivariate EDA |

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

x = np.linspace(0, 10, 50)
axes[0, 0].plot(x, np.sin(x), linewidth=2)
axes[0, 0].set_title('Line Plot\n"Trend over time"')

cats = ['A', 'B', 'C', 'D']
vals = [23, 45, 12, 38]
axes[0, 1].bar(cats, vals, color='coral')
axes[0, 1].set_title('Bar Chart\n"Compare categories"')

axes[0, 2].hist(rng.normal(50, 15, 500), bins=25, color='seagreen', edgecolor='white')
axes[0, 2].set_title('Histogram\n"Distribution"')

axes[1, 0].scatter(rng.normal(0, 1, 100), rng.normal(0, 1, 100), alpha=0.5, c='steelblue')
axes[1, 0].set_title('Scatter Plot\n"Relationship"')

box_data = [rng.normal(loc, 5, 50) for loc in [20, 25, 30, 22]]
axes[1, 1].boxplot(box_data, labels=['Q1', 'Q2', 'Q3', 'Q4'])
axes[1, 1].set_title('Box Plot\n"Distribution + Outliers"')

hm_data = rng.random((5, 5))
im = axes[1, 2].imshow(hm_data, cmap='YlOrRd', aspect='auto')
axes[1, 2].set_title('Heatmap\n"Matrix of values"')
plt.colorbar(im, ax=axes[1, 2])

fig.suptitle('Plot Type Cheat Sheet', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Styling — Making Publication-Quality Plots

Default plots are ugly. A few tweaks make them professional.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.linspace(0, 10, 100)

axes[0].plot(x, np.sin(x))
axes[0].plot(x, np.cos(x))
axes[0].set_title('Default — Ugly')

axes[1].plot(x, np.sin(x), color='#e74c3c', linewidth=2.5, label='sin(x)')
axes[1].plot(x, np.cos(x), color='#3498db', linewidth=2.5, linestyle='--', label='cos(x)')
axes[1].set_title('Styled — Professional', fontsize=13, fontweight='bold')
axes[1].set_xlabel('x', fontsize=11)
axes[1].set_ylabel('y', fontsize=11)
axes[1].legend(fontsize=10, framealpha=0.9)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(0, 10)
axes[1].set_ylim(-1.3, 1.3)

plt.tight_layout()
plt.show()

---
## Key Takeaways

| Concept | What to Remember |
|---------|------------------|
| **API** | Always use `fig, ax = plt.subplots()` |
| **Line** | Trends. Use `ax.plot()` |
| **Bar** | Categories. Vertical or horizontal |
| **Histogram** | Distribution. Adjust `bins` |
| **Scatter** | Relationships. Color/size for extra dims |
| **Box/Violin** | Compare distributions across groups |
| **Heatmap** | Correlation matrices — use Seaborn |
| **Seaborn** | 1 line = 10 lines of Matplotlib |
| **Style** | Remove top/right spines, light grid, bold title |